[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataguirre/Curso-IA-Aplicada/blob/main/Semana%2009-10_%20NLP_y_etica/fundamentos_nlp.ipynb)



# Fundamentos de NLP

In [ ]:
!pip install gensim
!pip install pandarallel

In [ ]:
import re
import nltk
import pandas as pd
from tqdm import tqdm
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# --- Descargas necesarias ---
nltk.download("stopwords")

# Base de datos Banco de la Republica

In [ ]:
banrep = pd.read_csv('https://drive.google.com/uc?id=1vz3530nztdYPs-x0ZELlRPjNEcK453hc')
banrep

In [ ]:
columns = ['uuid', 'name', 'collection', 'authors', 'type', 'date', 'abstract_spa',
           'doi_link', 'access_rights', 'access_rights', 'keywords_spa', 'jel_spa']

banrep = banrep[(~banrep['abstract_spa'].isna()) & (~banrep['authors'].isna())][columns].reset_index(drop=True)
banrep

# Preprocesamiento de texto

### Algunos operadores

<div> <center>

| **RE** | **Expansión** |                          **Patrón capturado**                         |
|:------:|:-------------:|:---------------------------------------------------------------------:|
|   \d   |     [0-9]     |                            Cualquier dígito                           |
|   \D   |     [^0-9]    |                          Cualquier no dígito                          |
|   \w   |  [a-zA-Z0-9_] |                  Cualquier alfanumérico o guion bajo                  |
|   \W   |     [ˆ\w]     |                 Cualquier no alfanumérico o guion bajo                |
|   \s   |  [ \r\t\n\f]  |                           Espacio en blanco                           |
|   \S   |     [ˆ\s]     |                        No un espacio en blanco                        |
|    *   |               |         Cero o más ocurrencias del caracter o expresión pasada        |
|    +   |               |         Una o más ocurrencias del caracter o expresión pasada         |
|    ?   |               | Exactamente cero o una ocurrencia del del caracter o expresión pasada |
|   {n}  |               |            *n* ocurrencias del caracter o expresión pasada            |
|  {n,m} |               |        De *n* a *m* ocurrencias del caracter o expresión pasada       |
|  {n,}  |               |      Por lo menos *n* ocurrencias del caracter o expresión pasada     |
|  {,m}  |               |         Hasta *m* ocurrencias del caracter o expresión pasada         |

</div> </center>

In [ ]:
# --- Recursos globales ---
STOPWORDS_ES = set(stopwords.words("spanish")) | {
    "doi", "http", "https", "www", "et", "al", "etc"
}
STEMMER_ES = SnowballStemmer("spanish")

# Mapa para quitar acentos (manteniendo ñ)
_ACCENT_MAP = str.maketrans("áéíóúüÁÉÍÓÚÜ", "aeiouuAEIOUU")

# Expresiones regulares útiles
_URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
_EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
_NUM_SYM = re.compile(r"[0-9]+|[_/#%€$£°ºª©™®§]")
_TOKEN_RE = re.compile(r"[a-zñ]+")

In [ ]:
def normalizacion(texto):
    """
    Normaliza texto en español:
    - Minúsculas
    - Quita URLs, emails, números y símbolos
    - Elimina tildes y diéresis (mantiene ñ)
    - Colapsa espacios múltiples
    """
    if not isinstance(texto, str):
        return ""
    t = texto.strip().lower()
    t = _URL_RE.sub(" ", t)
    t = _EMAIL_RE.sub(" ", t)
    t = _NUM_SYM.sub(" ", t)
    t = t.translate(_ACCENT_MAP)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def tokenizar_texto(texto):
    """
    Tokeniza extrayendo palabras (incluye ñ).
    """
    t = normalizacion(texto)
    return _TOKEN_RE.findall(t)

def eliminar_stopwords(tokens):
    """
    Elimina stopwords en español.
    """
    return [w for w in tokens if w not in STOPWORDS_ES and len(w) > 1]

def stemmizar(tokens):
    """
    Aplica stemming en español usando SnowballStemmer.
    """
    return [STEMMER_ES.stem(w) for w in tokens]

def preprocesamiento(texto):
    """
    Pipeline completo: normaliza → tokeniza → quita stopwords → stemming.
    Devuelve lista de tokens procesados.
    """
    tokens = tokenizar_texto(texto)
    tokens = eliminar_stopwords(tokens)
    tokens = stemmizar(tokens)
    return tokens

In [ ]:
ejemplo = banrep['abstract_spa'].iloc[0]
print("Texto original: \n")
print(ejemplo)

In [ ]:
# Normalización
ejemplo_norm = normalizacion(ejemplo)
print("Texto normalizado:")
print(ejemplo_norm, "\n")

In [ ]:
# Tokenización
ejemplo_tokens = tokenizar_texto(ejemplo)
print("Tokens:")
print(' '.join(ejemplo_tokens), "\n")

In [ ]:
# Eliminación de stopwords
ejemplo_nosw = eliminar_stopwords(ejemplo_tokens)
print("Sin stopwords:")
print(' '.join(ejemplo_nosw), "\n")

In [ ]:
# Stemming
ejemplo_stem = stemmizar(ejemplo_nosw)
print("Con stemming:")
print(' '.join(ejemplo_stem))

In [ ]:
tqdm.pandas(desc="Preprocesando abstracts ES")
banrep["tokens_clean"] = (
    banrep["abstract_spa"]
    .fillna("")
    .astype(str)
    .progress_apply(preprocesamiento)
)

**Paralelizacion?**

La **paralelización** consiste en dividir un conjunto de tareas en varios procesos que se ejecutan **simultáneamente** en distintos núcleos del procesador (CPUs).  
En lugar de procesar cada texto de forma secuencial con un solo hilo, el sistema reparte el trabajo entre varios *workers* que operan en paralelo, reduciendo significativamente el tiempo total de ejecución.

En tareas como el **preprocesamiento de texto**, esta técnica resulta especialmente eficaz porque cada observación (por ejemplo, cada *abstract*) se puede limpiar o tokenizar **de manera independiente** del resto.  
Esa independencia permite distribuir las operaciones entre múltiples núcleos sin riesgo de conflictos o dependencia entre procesos.

En este ejemplo, la librería `pandarallel` divide el `DataFrame` en fragmentos, los envía a los *workers* disponibles y combina los resultados al finalizar, mostrando además una barra de progreso.  
Gracias a esto, el preprocesamiento se completa **en una fracción del tiempo** comparado con el uso tradicional de `apply` o `progress_apply`.

En resumen:  
- Aprovecha todos los núcleos disponibles del CPU.  
- Reduce tiempos en tareas repetitivas e independientes.  
- Es ideal para pipelines de limpieza o tokenización de texto.

In [ ]:
from pandarallel import pandarallel
import os
import time

print(f'CPUs: {os.cpu_count()}')

In [ ]:
workers = 2
pandarallel.initialize(progress_bar=True, nb_workers=workers)

start_time = time.time()

banrep["tokens_clean"] = (
    banrep["abstract_spa"]
    .fillna("")
    .astype(str)
    .parallel_apply(preprocesamiento)
)

end_time = time.time()
elapsed = end_time - start_time

print(f"Preprocesamiento completado en {elapsed:.2f} segundos")

# Nube de Palabras

Con nuestro texto normalizado, podemos hacer un análisis sencillo del texto usando nubes de palabras. Las nubes de palabras (también conocidas como "word clouds" en inglés) son representaciones visuales de un conjunto de palabras en un texto, donde el tamaño de cada palabra se determina en función de su frecuencia de aparición en el texto. Es una forma popular y efectiva de visualizar la distribución y relevancia de las palabras en un documento.

La librería `WordCloud` en `Python` es una herramienta ampliamente utilizada para crear nubes de palabras de manera sencilla. La función principal de `WordCloud` es tomar un texto y generar una nube de palabras donde el tamaño de cada palabra se determina por su frecuencia en el texto.

In [ ]:
tqdm.pandas(desc="Preprocesando abstracts ES")
banrep["tokenizar"] = (
    banrep["abstract_spa"]
    .fillna("")
    .astype(str)
    .progress_apply(tokenizar_texto)
)
docs_tokens = banrep['tokenizar'].dropna().tolist()
text = " ".join(" ".join(doc) for doc in docs_tokens)

In [ ]:
# 3) Crear y mostrar la nube
wc = WordCloud(
    width=1600, height=800,
    background_color="white",
    stopwords=STOPWORDS_ES,
    collocations=False  # evita juntar palabras comunes que no quieres
).generate(text)

plt.figure(figsize=(18,9))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS
from gensim.models.phrases import Phrases, Phraser
from tqdm import tqdm


docs = banrep["tokenizar"].dropna().tolist()

# ------------------------------------------------------------
# Detectar bigramas frecuentes con Gensim
# ------------------------------------------------------------
# min_count: número mínimo de veces que un par debe aparecer
# threshold: entre más alto, menos bigramas (ajusta según corpus)
bigram_model = Phrases(docs, min_count=10, threshold=20)
bigram_phraser = Phraser(bigram_model)

# Aplicar el modelo a todos los documentos
docs_with_bigrams = [bigram_phraser[doc] for doc in tqdm(docs, desc="Detectando bigramas")]

# ------------------------------------------------------------
# Aplanar todo el corpus a una sola lista de tokens
# ------------------------------------------------------------
tokens_flat = [token for doc in docs_with_bigrams for token in doc]

# Opcional: eliminar tokens muy cortos o numéricos
tokens_flat = [t for t in tokens_flat if len(t) > 2 and not t.isnumeric()]

# ------------------------------------------------------------
# Generar el texto completo para la nube
# ------------------------------------------------------------
text = " ".join(tokens_flat)

wc = WordCloud(
    width=1600,
    height=800,
    background_color="white",
    stopwords=STOPWORDS_ES,
    collocations=False,   # evita duplicar bigramas automáticos
    max_words=200
).generate(text)

plt.figure(figsize=(18,9))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.show()

# TF-IDF

$$
tf-idf_{ij}=tf_{ij} \times \left( \log \left( \frac{1+N}{1+df_i} \right)+1\right)
$$

donde:

- $tf_{ij}$ es la frecuencia palabra $i$ en el documento $j$
- $df_{ij}$ es el número de documentos que contienen la palabra $i$
- $N$ es el número de documentos

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def tokenizer_es(x):
    return preprocesamiento(x)

vectorizer = TfidfVectorizer(
    tokenizer=tokenizer_es,
    preprocessor=lambda x: x,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.85,
    sublinear_tf=True,
    dtype=np.float32
)

X_tfidf = vectorizer.fit_transform(banrep["abstract_spa"].fillna("").astype(str))
feat_names = vectorizer.get_feature_names_out()
print("Dimensión TF-IDF:", X_tfidf.shape)

In [ ]:
X_tfidf

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import TfidfModel
from gensim import matutils
import numpy as np

# Hacemos nuestro corpus de documentos en una lista
texts = banrep["tokens_clean"].tolist()

# Creamos un diccionario de palabras unicas
dictionary = Dictionary(texts)

# Ajusta estos umbrales a tu tamaño de corpus
dictionary.filter_extremes(no_below=5, no_above=0.85)  # >=5 docs, <=85% de docs
dictionary.compactify()

# BoW (lista de listas de (term_id, count))
corpus_bow = [dictionary.doc2bow(doc) for doc in texts]

print(dictionary.token2id)

In [ ]:
# 4) TF-IDF con Gensim (streaming-friendly)
#    smartirs='ntc' aplica una variante estándar de normalización
tfidf_model = TfidfModel(corpus_bow, smartirs='ntc', normalize=True)
corpus_tfidf = tfidf_model[corpus_bow]  # iterable de listas (term_id, tfidf)

# 5) OPCIONAL: convertir a matriz SciPy CSR para ML downstream
#    (filas = documentos, columnas = términos)
X_tfidf_csr = matutils.corpus2csc(corpus_tfidf, num_terms=len(dictionary)).T.tocsr()

print("Dimensión TF-IDF:", X_tfidf_csr.shape)  # (n_docs, n_terms)

# 6) OPCIONAL: nombres de características (vocabulario)
feature_names = [dictionary[i] for i in range(len(dictionary))]

# 7) Ejemplo: top-10 términos por documento 0
doc_id = 0
row = X_tfidf_csr.getrow(doc_id)
top_idx = np.argsort(row.data)[::-1][:10]
top_terms = [(feature_names[row.indices[i]], float(row.data[i])) for i in top_idx]
print("Top términos doc 0:", top_terms)

# Distancia del coseno

In [ ]:
from gensim.similarities import MatrixSimilarity
import numpy as np

# Crear índice de similitud TF-IDF
index = MatrixSimilarity(corpus_tfidf, num_features=len(dictionary))

# Definir función de búsqueda
def buscar_documentos(query, top_n=5):
    """
    Busca en el corpus los documentos más similares a la query.
    Imprime título, similitud y fragmento del abstract.
    """
    # Preprocesar la consulta (igual que los textos del corpus)
    tokens = preprocesamiento(query)
    bow = dictionary.doc2bow(tokens)
    tfidf_vec = tfidf_model[bow]

    # Calcular similitud coseno entre query y documentos
    sims = index[tfidf_vec]  # vector de similitudes

    # Top-N documentos más similares
    top_idx = np.argsort(sims)[::-1][:top_n]
    resultados = []
    for i in top_idx:
        print(f"{banrep.loc[i, 'name']}")
        print(f"   Similitud: {float(sims[i]):.4f}")
        print(f"   Extracto: {banrep.loc[i, 'abstract_spa'][:600].strip()}\n")
        print(f"   UUID: {banrep.loc[i, 'uuid']}\n")
        print("-" * 100)

        resultados.append({
            "id": i,
            "titulo": banrep.loc[i, "name"],
            "similitud": float(sims[i]),
            "texto": banrep.loc[i, "abstract_spa"][:600].strip(),
            "link": banrep.loc[i, "doi_link"],
            "uuid": banrep.loc[i, "uuid"]
        })
    return resultados

In [ ]:
# impacto de la política monetaria en la inflación colombiana
# evaluacion de impacto
# economía de Suroccidente
# balanza de pagos
# evolucion del endeudamiento externo
# consumo de los hogares

from ipywidgets import interact, Text, IntSlider

def demo_busqueda(q, k):
    res = buscar_documentos(q, top_n=k)

interact(
    demo_busqueda,
    q=Text(value="regla fiscal", description="Query:"),
    k=IntSlider(value=5, min=1, max=15, step=1, description="Top-N")
);

# Machine learning

In [ ]:
banrep[banrep['uuid'] == '54b8488b-c6e8-4bd1-837f-76ec68d7adec']['abstract_spa'].iloc[0]

In [ ]:
banrep[banrep['uuid'] == '54b8488b-c6e8-4bd1-837f-76ec68d7adec']['tokens_clean'].iloc[0]

In [ ]:
banrep